# 03 · MobileNetV2 transfer learning  *(Onkar - part 2a)*

**Idea:** start from a network that already learned general visual features on ImageNet and adapt it to six fruit classes. Two training stages:

| Stage | What trains | Learning rate | Why |
|---|---|---|---|
| **1 · head** | only the new classifier (GAP → dropout → softmax) | 1e-3 | the new layers start random; keep the pretrained features intact while they learn |
| **2 · fine-tune** | the top 40 backbone layers + head | 2e-5 | gently adapt high-level features to fruit; **BatchNorm layers stay frozen** |

The backbone is always run in *inference mode* so BatchNorm statistics are not corrupted by small batches. **Use a GPU runtime** (Runtime → Change runtime type → T4).

In [ ]:
# ============ SETUP - run this first after every fresh Colab runtime ============
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # e.g. "https://github.com/<your-username>/fruit-freshness-classifier.git"
                # leave "" if you uploaded/unzipped the project folder yourself

IN_COLAB = "google.colab" in sys.modules
PROJECT_DIR = Path("/content/fruit-freshness-classifier") if IN_COLAB else Path.cwd().parent
if not (PROJECT_DIR / "src").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        raise SystemExit(f"Project folder not found at {PROJECT_DIR}. Set REPO_URL above or upload the project there.")
elif REPO_URL and IN_COLAB:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

if os.getenv("FRUIT_SKIP_PIP") != "1":     # only kagglehub is needed; Colab already has TensorFlow etc.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import json
import matplotlib.pyplot as plt
import pandas as pd
from fruitfresh import config
from fruitfresh.colab import setup_workspace, gpu_summary

WORK_DIR = setup_workspace()      # Colab: Google Drive (survives runtime resets)
SEED = config.SEED
print(gpu_summary())
print("Project folder :", PROJECT_DIR)
print("Work folder    :", WORK_DIR, "(models, histories, results are saved here)")

In [ ]:
from fruitfresh.data_index import get_dataset_root
from fruitfresh.splits import load_split

DATASET_ROOT = get_dataset_root()
SPLIT_DIR = WORK_DIR / "data" / "splits"
if not SPLIT_DIR.exists():
    SPLIT_DIR = PROJECT_DIR / "data" / "splits"      # the committed copy of the team split
train_df, val_df, _ = load_split(SPLIT_DIR)           # the TEST split is deliberately ignored (`_`)
print(f"train {len(train_df)} images / {train_df.source_id.nunique()} photos | "
      f"validation {len(val_df)} images / {val_df.source_id.nunique()} photos")
print("Split folder:", SPLIT_DIR)

In [ ]:
PRETRAINED = os.getenv("FRUIT_NO_PRETRAINED", "0") != "1"   # keep True: ImageNet weights are the point

## 1 · The two stages, before training
Check that the freezing does what you expect: stage 1 must have **0** trainable backbone layers; stage 2 only the top layers, with **0 trainable BatchNorm** layers.

In [ ]:
from fruitfresh.models import build_model, set_trainable_stage
m = build_model("mobilenet_v2", pretrained=PRETRAINED)
ft_layers = config.MODEL_SPECS["mobilenet_v2"]["ft_layers"]
stages = {"stage 1 (head)": set_trainable_stage(m, "head"), "stage 2 (fine-tune)": set_trainable_stage(m, "finetune", ft_layers)}
display(pd.DataFrame(stages).T)
del m

## 2 · Train stage 1 → stage 2
Finished stages are detected and skipped (stage 1 is saved as `stage1.keras`), so you can resume after a disconnect.

In [ ]:
from fruitfresh.train import train_model, run_dir_for
FORCE = False
info = train_model("mobilenet_v2", train_df, val_df, DATASET_ROOT, WORK_DIR, seed=SEED, force=FORCE, pretrained=PRETRAINED)
display(pd.DataFrame(info["stages"]).T)
pd.Series({k: info[k] for k in ("epochs_run", "best_epoch", "best_val_loss", "best_val_accuracy", "params", "training_seconds")})

In [ ]:
from fruitfresh.viz import plot_history
RUN_DIR = run_dir_for(WORK_DIR, "mobilenet_v2", SEED)
fig = plot_history(RUN_DIR / "history.csv", "MobileNetV2 (dashed line = start of fine-tuning)"); plt.show()

**What to expect:** validation accuracy is already good after stage 1; stage 2 should improve it a little or keep it. If stage 2 makes validation clearly *worse*, the fine-tuning learning rate is too high - that is experiment `ft_lr_1e-5` in notebook 04.

## 3 · Verify what fine-tuning really changed
Compares the backbone weights of the stage-1 model and the final model. Expected: changes only in the **last layers**, and **0 BatchNorm layers changed**.

In [ ]:
from fruitfresh.models import verify_finetuning
verify_finetuning(RUN_DIR / "stage1.keras", RUN_DIR / "model.keras")

## 4 · Validation diagnostics

In [ ]:
from fruitfresh.evaluate import evaluate_validation, plot_confusion_matrix
m = evaluate_validation(RUN_DIR, val_df, DATASET_ROOT)
print(f"validation: accuracy {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} ({m['n_images']} images, {m['n_sources']} photos)")
display(pd.DataFrame(m["per_class"]).T.round(3))
fig = plot_confusion_matrix(m["confusion_matrix"], title="MobileNetV2 - validation"); plt.show()

## 5 · Quick look next to the baseline (validation only)
This is a progress check on **your own two runs**, not the final comparison - the official test-set comparison with confidence intervals is Rucha's part.

In [ ]:
rows = []
for run in ("baseline_cnn", "mobilenet_v2"):
    rd = run_dir_for(WORK_DIR, run, SEED)
    i = json.loads((rd / "train_info.json").read_text()); v = json.loads((rd / "val_eval" / "metrics.json").read_text())
    rows.append({"run": run, "params (M)": round(i["params"] / 1e6, 2), "val accuracy": round(v["accuracy"], 4), "val macro-F1": round(v["macro_f1"], 4)})
pd.DataFrame(rows)

## Checklist before notebook 04
- [ ] stage reports show 0 trainable BatchNorm layers; `verify_finetuning` shows 0 BatchNorm layers changed
- [ ] `runs/mobilenet_v2/seed42/` has `stage1.keras`, `model.keras`, `history.csv`, `train_info.json`
- [ ] you can explain in your own words why stage 1 comes before stage 2